In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest
from langchain.agents.middleware.types import ModelResponse
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, ToolMessage
from typing import Any, Callable


class Calculatorinput(BaseModel):
    """Input for calculator"""
    expression: str = Field(description="The mathematical expression to evaluate.")

class Searchinput(BaseModel):
    """Input for search"""
    query: str = Field(description="searchin on web") 



@tool(args_schema=Calculatorinput)
def calculator(expression: str) -> str:
    """Perform mathematical calculations"""
    try:
        result = eval(expression, {"__builltins__": {}}, {})
        return f"The result is: {result}"

    except Exception as e:
        return f"Error: {e}"

@tool(args_schema=Searchinput)
def search(query: str) -> str:
    """Search for information. May fail occasionally for demonstration."""

    if "error" in query.lower():
        raise Exception("search service temporarily not available")

    return f"search result for '{query}'.Found relavent information about {query}"



# Middleware 1: Dynamic Model Selection
# Switches to a more capable model for complex conversations
class DynamicModelMiddleware(AgentMiddleware):
    """Switch to a more capable model for complex conversations."""

    def __init__(self, messages_threshold: int = 10):
        super().__init__()
        self.messages_threshold = messages_threshold


    def wrap_model_call(
            self, 
            request:ModelRequest, 
            handler: Callable[[ModelRequest], ModelResponse]
            ) -> ModelResponse:
        message_count = len(request.state['messages'])
        print(f"Middleware: message count {message_count}")

        # For complex conversations (>threshold messages), could use more capable model
        if message_count > self.messages_threshold:
            print("Middleware would switch to more capable model")
            # In production: request = request.override(model=advanced_model)

        print("  [Middleware] ✓ Using current model")
        return handler(request) 


# Middleware 2: Tool Error Handler
# Catches tool failures and provides helpful fallback messages
class ToolErrorMiddleware(AgentMiddleware):
    """Catch tool failures and provide graceful fallbacks."""

    def wrap_tool_call(
            self,
            request: Any, 
            handler: Callable[[Any], ToolMessage]
            ) -> ToolMessage:
        try:
            return handler(request)

        except Exception as e:
            tool_name = request.tool_call.get("name", "unknown")
            print(f"Middleware tool {tool_name} failed: {e}")
            print("Middleware returning fallback message")
            # Return graceful fallback instead of crashing

            return ToolMessage(
                content=f"I encountered an error while using the {tool_name} tool. "
                f"Let me try a different approach to answer your question.",
                tool_call_id=request.tool_call.get("id", ""),
            )            



def main():
    print("🔧 Agent with Middleware Example\n")

    # Create the model
    model = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0

    )

    # Create agent with middleware
    agent = create_agent(
        model,
        tools=[calculator, search],
        middleware=[
            DynamicModelMiddleware(messages_threshold=10),
            ToolErrorMiddleware(),
        ],
    )

    # Test 1: Simple calculation (middleware logs but doesn't change behavior)
    print("Test 1: Simple calculation")
    print("─" * 60)
    query1 = "What is 25 * 8?"
    print(f"👤 User: {query1}\n")
    response1 = agent.invoke({"messages": [HumanMessage(content=query1)]})
    last_message1 = response1["messages"][-1]
    print(f"\n🤖 Agent: {last_message1.content}\n\n")

    # Test 2: Search with error handling (triggers error middleware)
    print("Test 2: Search with error handling")
    print("─" * 60)
    query2 = "Search for information about error handling"
    print(f"👤 User: {query2}\n")
    response2 = agent.invoke({"messages": [HumanMessage(content=query2)]})
    last_message2 = response2["messages"][-1]
    print(f"\n🤖 Agent: {last_message2.content}\n\n")

    print("💡 Middleware Benefits:")
    print("   • Dynamic model selection → Cost optimization")
    print("   • Error handling → Graceful degradation")
    print("   • Logging → Easy debugging")
    print("   • Flexibility → Customize behavior without changing tools\n")

    print("✅ Production Use Cases:")
    print("   • Switch to cheaper models for simple queries")
    print("   • Automatic retries with exponential backoff")
    print("   • Request/response logging for monitoring")
    print("   • User context injection (auth, permissions)")
    print("   • Rate limiting and quota management")


if __name__ == "__main__":
    main()

🔧 Agent with Middleware Example

Test 1: Simple calculation
────────────────────────────────────────────────────────────
👤 User: What is 25 * 8?

Middleware: message count 1
  [Middleware] ✓ Using current model

🤖 Agent: 25 × 8 = 200.


Test 2: Search with error handling
────────────────────────────────────────────────────────────
👤 User: Search for information about error handling

Middleware: message count 1
  [Middleware] ✓ Using current model
Middleware tool search failed: search service temporarily not available
Middleware returning fallback message
Middleware: message count 3
  [Middleware] ✓ Using current model
Middleware tool search failed: search service temporarily not available
Middleware returning fallback message
Middleware: message count 5
  [Middleware] ✓ Using current model

🤖 Agent: Below is a concise yet comprehensive overview of **error handling**—what it is, why it matters, the main strategies used across programming languages, common patterns, and best‑practice rec